---
title: A Simple Machine Learning Example
jupyter: python3
---

In this notebook we train a simple machine-learning model to **predict the value of a house from the median income of its neighborhood**. We'll show the data to two different kinds of models, have them make predictions on houses they've never seen, and measure how far off they are. Run the cells from top to bottom.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

sns.set_theme(style="whitegrid")

The data loads straight from the web - just run the cell below, there's nothing to download.

In [ ]:
housing = pd.read_csv("https://drive.google.com/uc?export=download&id=1QFm_7cTiYtGDJkxwhPzXKvqYjTnPoHLN")
housing.head()

## Pick a feature and a target

The **feature** is what we learn *from* (median income); the **target** is what we want to predict (median house value). We put the feature in `X` and the target in `y`. (The double brackets around `X` keep it a 2-D table, which is the shape scikit-learn expects.)

In [ ]:
X = housing[["median_income"]]
y = housing["median_house_value"]

## Split into training and test sets

We never test a model on the same data it learned from - that would be like grading students on the exact homework they already saw. So we split the data: most of it to **train** on, and a held-back chunk to **test** on.

Notice we split *randomly* (`train_test_split` shuffles first). That matters: if the rows were in some order - say, sorted by location - taking the "first 80%" could hand the model a biased slice and quietly skew everything. Random splitting guards against that.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training on", len(X_train), "houses; testing on", len(X_test), "houses.")

## Train a linear regression

A **linear regression** finds the straight line that best fits the training data. We `fit` it on the training set, then ask it to `predict` values for the test houses it has never seen.

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

predictions = lin_reg.predict(X_test)

# compare the first few predictions to the real values
pd.DataFrame({"actual": y_test.values[:5], "predicted": predictions[:5].round(0)})

How far off is it, on average? The **RMSE** (root mean squared error) is roughly the typical size of the model's mistakes, in dollars.

In [ ]:
rmse = mean_squared_error(y_test, predictions) ** 0.5
print(f"Linear regression is off by about ${rmse:,.0f} on a typical house.")

Let's see the line it learned, drawn through the test data:

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
sns.scatterplot(x=X_test["median_income"], y=y_test, alpha=0.1, ax=ax)
ax.plot(X_test["median_income"], predictions, color="red", linewidth=2, label="model's line")
ax.set(title="Predicting house value from income",
       xlabel="Median Income", ylabel="Median House Value ($)")
ax.legend()

## Try a different model: a decision tree

A **decision tree** learns a series of yes/no questions instead of a straight line. Same data, same train/test split - let's see if it does better or worse.

In [ ]:
tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(X_train, y_train)
tree_predictions = tree_reg.predict(X_test)

tree_rmse = mean_squared_error(y_test, tree_predictions) ** 0.5
print(f"Linear regression:  off by about ${rmse:,.0f}")
print(f"Decision tree:      off by about ${tree_rmse:,.0f}")

Both models are using only *one* clue (income) to guess a house's value, so neither is amazing - real estate depends on far more than income. That's the natural next question: which other columns would help the model predict better?